# Лабораторная работа 6

Линейное уравнение переноса

Решается уравнение

$$
\frac{\partial U}{\partial t} + a \frac{\partial U}{\partial x} = f(x, t)
$$

Исходные данные:

- $U(x, 0) = 2x^2 - 5x + 5$
- $U(0, t) = t^2 - 5t + 5$ для $a > 0$
- $U(1, t) = t^2 - 5t + 2$ для $a < 0$
- $f(x, t) = 3x$

Шаги сетки: $h_x = 0.1$, $h_t = 0.05$.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DX = 0.1
DT = 0.05
T_END = 10.0
X_RECT = np.arange(0.0, 1.0 + DX / 2, DX)
T_GRID = np.arange(0.0, T_END + DT / 2, DT)
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

def u0(x):
    return 2 * x**2 - 5 * x + 5

def u_left(t):
    return t**2 - 5 * t + 5

def u_right(t):
    return t**2 - 5 * t + 2

def f(x, t):
    return 3 * x

def exact_halfplane(x, t, a):
    xi = x - a * t
    return u0(xi) + 3 * x * t - 1.5 * a * t**2


In [2]:
def solve_halfplane(a):
    if a > 0:
        x_full = np.arange(-20.0, 1.0 + DX / 2, DX)
        scheme_name = "Схема 1 (явная, a > 0)"
    else:
        x_full = np.arange(0.0, 21.0 + DX / 2, DX)
        scheme_name = "Схема 2 (явная, a < 0)"

    u = np.zeros((len(T_GRID), len(x_full)))
    u[0, :] = u0(x_full)

    for n in range(len(T_GRID) - 1):
        t_prev = T_GRID[n]
        t_next = T_GRID[n + 1]
        c = a * DT / DX
        if a > 0:
            u[n + 1, 0] = exact_halfplane(x_full[0], t_next, a)
            for i in range(1, len(x_full)):
                u[n + 1, i] = u[n, i] - c * (u[n, i] - u[n, i - 1]) + DT * f(x_full[i], t_prev)
        else:
            u[n + 1, -1] = exact_halfplane(x_full[-1], t_next, a)
            for i in range(len(x_full) - 2, -1, -1):
                u[n + 1, i] = u[n, i] - c * (u[n, i + 1] - u[n, i]) + DT * f(x_full[i], t_prev)

    mask = (x_full >= 0.0 - 1e-12) & (x_full <= 1.0 + 1e-12)
    return {"domain": "Полуплоскость", "a": a, "scheme": scheme_name, "x": x_full[mask], "t": T_GRID, "u": u[:, mask]}

def solve_rectangle_explicit(a):
    u = np.zeros((len(T_GRID), len(X_RECT)))
    u[0, :] = u0(X_RECT)
    c = a * DT / DX

    for n in range(len(T_GRID) - 1):
        t_prev = T_GRID[n]
        t_next = T_GRID[n + 1]
        if a > 0:
            u[n + 1, 0] = u_left(t_next)
            for i in range(1, len(X_RECT)):
                u[n + 1, i] = u[n, i] - c * (u[n, i] - u[n, i - 1]) + DT * f(X_RECT[i], t_prev)
            scheme_name = "Схема 1 (явная, a > 0)"
        else:
            u[n + 1, -1] = u_right(t_next)
            for i in range(len(X_RECT) - 2, -1, -1):
                u[n + 1, i] = u[n, i] - c * (u[n, i + 1] - u[n, i]) + DT * f(X_RECT[i], t_prev)
            scheme_name = "Схема 2 (явная, a < 0)"

    return {"domain": "Прямоугольник", "a": a, "scheme": scheme_name, "x": X_RECT, "t": T_GRID, "u": u}


In [3]:
def solve_rectangle_implicit_upwind():
    a = 2.0
    c = a * DT / DX
    u = np.zeros((len(T_GRID), len(X_RECT)))
    u[0, :] = u0(X_RECT)

    for n in range(len(T_GRID) - 1):
        t_next = T_GRID[n + 1]
        u[n + 1, 0] = u_left(t_next)
        for i in range(1, len(X_RECT)):
            rhs = u[n, i] + DT * f(X_RECT[i], t_next)
            u[n + 1, i] = (rhs + c * u[n + 1, i - 1]) / (1 + c)

    return {"domain": "Прямоугольник", "a": a, "scheme": "Схема 3 (неявная, a > 0)", "x": X_RECT, "t": T_GRID, "u": u}

def solve_rectangle_box(a):
    c = a * DT / (2 * DX)
    u = np.zeros((len(T_GRID), len(X_RECT)))
    u[0, :] = u0(X_RECT)

    for n in range(len(T_GRID) - 1):
        t_next = T_GRID[n + 1]
        if a > 0:
            u[n + 1, 0] = u_left(t_next)
            for i in range(1, len(X_RECT)):
                rhs = u[n, i] + c * u[n, i - 1] - c * u[n, i] + c * u[n + 1, i - 1] + DT * f((X_RECT[i] + X_RECT[i - 1]) / 2, t_next)
                u[n + 1, i] = rhs / (1 + c)
        else:
            u[n + 1, -1] = u_right(t_next)
            for i in range(len(X_RECT) - 2, -1, -1):
                rhs = u[n, i] - c * u[n, i + 1] + c * u[n, i] - c * u[n + 1, i + 1] + DT * f((X_RECT[i] + X_RECT[i + 1]) / 2, t_next)
                u[n + 1, i] = rhs / (1 - c)

    return {"domain": "Прямоугольник", "a": a, "scheme": "Схема 4 (неявная 'ящик')", "x": X_RECT, "t": T_GRID, "u": u}

def draw_case(case, filename):
    X, T = np.meshgrid(case["x"], case["t"])
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(X, T, case["u"], cmap="viridis", edgecolor="none")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_zlabel("U(x, t)")
    ax.set_title(f'{case["domain"]}, a = {case["a"]:.0f}, {case["scheme"]}')
    fig.colorbar(surf, shrink=0.6, aspect=12)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)


In [4]:
cases = [
    solve_halfplane(2.0),
    solve_halfplane(-2.0),
    solve_rectangle_explicit(2.0),
    solve_rectangle_explicit(-2.0),
    solve_rectangle_implicit_upwind(),
    solve_rectangle_box(2.0),
    solve_rectangle_box(-2.0),
]

for index, case in enumerate(cases, start=1):
    filename = f"plot_{index:02d}.png"
    draw_case(case, filename)
    print(f'{index}. {case["domain"]}, a = {case["a"]:.0f}, {case["scheme"]} -> {filename}')


1. Полуплоскость, a = 2, Схема 1 (явная, a > 0) -> plot_01.png
2. Полуплоскость, a = -2, Схема 2 (явная, a < 0) -> plot_02.png
3. Прямоугольник, a = 2, Схема 1 (явная, a > 0) -> plot_03.png
4. Прямоугольник, a = -2, Схема 2 (явная, a < 0) -> plot_04.png
5. Прямоугольник, a = 2, Схема 3 (неявная, a > 0) -> plot_05.png
6. Прямоугольник, a = 2, Схема 4 (неявная 'ящик') -> plot_06.png
7. Прямоугольник, a = -2, Схема 4 (неявная 'ящик') -> plot_07.png
